In [1]:
import os
import json
import copy
import math
import random
import warnings
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from joblib import dump, load

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data  
from torch_geometric.explain import Explainer, GNNExplainer  
from torch_geometric.nn import GATConv  

import shap 

from gprofiler import GProfiler  
import gseapy as gp  

from captum.attr import IntegratedGradients  


from IPython.display import display

from sklearn.decomposition import PCA as CPU_PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)


# Reproducibility
BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]

SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
RNG = np.random.default_rng(SEED)


def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    RNG = np.random.default_rng(SEED)


warnings.filterwarnings("ignore", category=UserWarning)
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {TORCH_DEVICE}")


# Paths
ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"
IN_META = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 8_taskaware_joint"
OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_CACHE = ARTIFACTS / "cache" / NOTEBOOK_SUBDIR
for d in [OUT_REPORTS, OUT_META, OUT_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

TASKAWARE_LOCK_FILENAME = "taskaware_joint_choice_prot.json"


# Backbone lock
LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(
        f"proteomics_backbone_lock.json not found at {LOCK_PATH}. "
        "Run the lock cell at the end of Notebook 3b first."
    )

with LOCK_PATH.open("r", encoding="utf-8") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM = backbone_lock["track2_stress_test_arm"]
DEPRIORITISED_ARM = backbone_lock.get("deprioritised_arm", "prot_rppa_ccle")

print("Backbone lock loaded.")
print(f"  Primary arm  : {PRIMARY_ARM}")
print(f"  Secondary arm: {SECONDARY_ARM}")
print(f"  Track 2 arm  : {TRACK2_ARM}")
print(f"  Deprioritised: {DEPRIORITISED_ARM}")
print(f"  Seeds to run : {ALL_SEEDS}")

ACTIVE_ARMS = sorted(list({PRIMARY_ARM, SECONDARY_ARM, TRACK2_ARM, DEPRIORITISED_ARM}))
FULL_FEATURE_SETS = [
    "rna",
    "cnv",
    "mut",
    "prot",
    "rna+cnv",
    "rna+mut",
    "rna+prot",
    "cnv+mut",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+mut",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]
ACTIVE_MODELS = ["joint_taskaware_attn"]

TASKAWARE_TEST_CONFIGS = [
    {"rank": rank, "arm": arm, "model": model, "feature_set": fs}
    for rank, (arm, model, fs) in enumerate(
        [
            (arm, model, fs)
            for arm in ACTIVE_ARMS
            for model in ACTIVE_MODELS
            for fs in FULL_FEATURE_SETS
        ],
        start=1,
    )
]
CONFIGS_BY_ARM = {
    arm: [cfg for cfg in TASKAWARE_TEST_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

EXPECTED_CONFIG_KEYS = {
    (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
    for cfg in TASKAWARE_TEST_CONFIGS
}
EXPECTED_CONFIG_KEYS_BY_ARM = {
    arm: {
        (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
        for cfg in cfgs
    }
    for arm, cfgs in CONFIGS_BY_ARM.items()
}

print("\nTask-aware joint configuration grid loaded.")
print("Active arms:", ACTIVE_ARMS)
print("Active models:", ACTIVE_MODELS)
print("Feature sets:", FULL_FEATURE_SETS)
display(pd.DataFrame(TASKAWARE_TEST_CONFIGS))

/home/andrija/Desktop/Final Year Project/FYP/.venv_captum/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch device: cuda
Backbone lock loaded.
  Primary arm  : prot_procan_depmapSanger
  Secondary arm: prot_ms_ccle_gygi
  Track 2 arm  : prot_combined_union
  Deprioritised: prot_rppa_ccle
  Seeds to run : [19537, 1584678, 17052356]

Task-aware joint configuration grid loaded.
Active arms: ['prot_combined_union', 'prot_ms_ccle_gygi', 'prot_procan_depmapSanger', 'prot_rppa_ccle']
Active models: ['joint_taskaware_attn']
Feature sets: ['rna', 'cnv', 'mut', 'prot', 'rna+cnv', 'rna+mut', 'rna+prot', 'cnv+mut', 'cnv+prot', 'mut+prot', 'rna+cnv+mut', 'rna+cnv+prot', 'rna+mut+prot', 'cnv+mut+prot', 'rna+cnv+mut+prot']


,rank,arm,model,feature_set
0,1,prot_combined_union,joint_taskaware_attn,rna
1,2,prot_combined_union,joint_taskaware_attn,cnv
2,3,prot_combined_union,joint_taskaware_attn,mut
3,4,prot_combined_union,joint_taskaware_attn,prot
4,5,prot_combined_union,joint_taskaware_attn,rna+cnv
5,6,prot_combined_union,joint_taskaware_attn,rna+mut
6,7,prot_combined_union,joint_taskaware_attn,rna+prot
7,8,prot_combined_union,joint_taskaware_attn,cnv+mut
8,9,prot_combined_union,joint_taskaware_attn,cnv+prot
9,10,prot_combined_union,joint_taskaware_attn,mut+prot


In [2]:
# Settings
N_DRUGS_BAKEOFF = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED = 10
PRIMARY_TARGET = "auc"

BASE_PCA_COMPONENTS = 100
PROT_MAX_INPUT_FEATURES = 2500
PROT_MIN_TRAIN_OBS_FRAC = 0.05

JOINT_LATENT_DIM = 128
JOINT_HIDDEN_DIM = 256
JOINT_DROPOUT = 0.10
JOINT_EPOCHS = 140
JOINT_PATIENCE = 18
JOINT_BATCH_SIZE = 64
JOINT_LR = 1e-3
JOINT_WEIGHT_DECAY = 1e-5
JOINT_PRED_LOSS_WEIGHT = 1.0
JOINT_RECON_LOSS_WEIGHT = 0.35
JOINT_REMASK_FRAC = 0.15
VAL_FRAC = 0.15
MIN_VAL_SAMPLES = 24

TASKAWARE_STRATEGIES = [
    {"name": "taskaware_joint", "add_indicators": False},
    {"name": "taskaware_joint+indicators", "add_indicators": True},
]

CACHE_VERSION = "v1_taskaware_joint"
CACHE_TAG = (
    f"{CACHE_VERSION}"
    f"__target{PRIMARY_TARGET}"
    f"__splits{N_SPLITS_DESIRED}"
    f"__ndrugs{N_DRUGS_BAKEOFF}"
    f"__mindrug{MIN_CELLS_PER_DRUG}"
    f"__basepca{BASE_PCA_COMPONENTS}"
    f"__protmax{PROT_MAX_INPUT_FEATURES}"
    f"__lat{JOINT_LATENT_DIM}"
    f"__hid{JOINT_HIDDEN_DIM}"
    f"__ep{JOINT_EPOCHS}"
    f"__bs{JOINT_BATCH_SIZE}"
    f"__lr{JOINT_LR}"
)

# Interpretability settings
INTERP_ARM = PRIMARY_ARM
INTERP_MODEL = ACTIVE_MODELS[0]
INTERP_FEATURE_SET = "rna+prot"
INTERP_DRUG = None
INTERP_FOLD = 0
INTERP_TOP_FEATURES_PER_MODALITY = {
    "rna": 250,
    "cnv": 150,
    "mut": 150,
    "prot": 400,
}
GNN_KNN_K = 8

In [3]:
# Helpers

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)


def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df


def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])


def safe_nanmean(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.mean()) if x.size > 0 else np.nan


def safe_nanmedian(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size > 0 else np.nan


def safe_nanstd(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    return float(x.std()) if x.size > 0 else np.nan


def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"


def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int,
    seed: int,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    groups = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        sp = GroupKFold(n_splits=n_splits)
        return (
            list(sp.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)),
            f"GroupKFold(n_splits={n_splits})",
        )
    n_splits = min(max(2, n_splits), n_cells)
    sp = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sp.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits})"


def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    if feature_set == "":
        return tuple()
    return tuple(feature_set.split("+"))


def _safe_name(s: str) -> str:
    s = str(s)
    return (
        s.replace(os.sep, "_")
         .replace(" ", "_")
         .replace("+", "plus")
         .replace(":", "_")
         .replace("/", "_")
         .replace("\\", "_")
    )


def _cache_digest(**parts) -> str:
    payload = json.dumps(parts, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:20]


def _cache_file(dirpath: Path, prefix: str, **parts) -> Path:
    dirpath.mkdir(parents=True, exist_ok=True)
    digest = _cache_digest(**parts)
    return dirpath / f"{prefix}__{digest}.joblib"


def checkpoint_matches_expected_grid(df: pd.DataFrame, expected_keys) -> bool:
    needed = {"config_rank", "arm", "model", "feature_set"}
    if not needed.issubset(df.columns):
        return False
    found = {
        (int(r["config_rank"]), str(r["arm"]), str(r["model"]).lower(), str(r["feature_set"]))
        for _, r in df[["config_rank", "arm", "model", "feature_set"]].drop_duplicates().iterrows()
    }
    return expected_keys.issubset(found)


def _split_train_val_idx(n_samples: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    idx = np.arange(n_samples)
    if n_samples < (MIN_VAL_SAMPLES * 2):
        return idx, np.array([], dtype=int)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_val = max(MIN_VAL_SAMPLES, int(round(n_samples * VAL_FRAC)))
    n_val = min(n_val, n_samples // 3)
    val_idx = np.sort(idx[:n_val])
    tr_idx = np.sort(idx[n_val:])
    return tr_idx, val_idx


def build_random_remask(obs_mask: np.ndarray, frac: float, rng: np.random.Generator) -> np.ndarray:
    obs_mask = np.asarray(obs_mask, dtype=bool)
    remask = np.zeros_like(obs_mask, dtype=bool)
    if frac <= 0.0:
        return remask
    for i in range(obs_mask.shape[0]):
        idx = np.flatnonzero(obs_mask[i])
        if len(idx) <= 1:
            continue
        n_mask = max(1, int(round(len(idx) * frac)))
        n_mask = min(n_mask, len(idx) - 1)
        chosen = rng.choice(idx, size=n_mask, replace=False)
        remask[i, chosen] = True
    return remask


def plot_bar(df: pd.DataFrame, label_col: str, value_col: str, title: str, out_path: Path, top_n: int = 15):
    tmp = df.head(top_n).copy()
    if tmp.empty:
        return
    plt.figure(figsize=(10, max(4, int(0.35 * len(tmp) + 1))))
    plt.barh(range(len(tmp)), tmp[value_col].to_numpy(dtype=float))
    plt.yticks(range(len(tmp)), tmp[label_col].astype(str).tolist())
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel(value_col)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close()


In [4]:
# Fold-safe transforms
class BaseModalityTransformer:
    def __init__(self, n_components: int, random_state: int):
        self.n_components = int(n_components)
        self.random_state = int(random_state)
        self._imp = SimpleImputer(strategy="median")
        self._scaler = StandardScaler()
        self._pca = None
        self._keep = None
        self.feature_names_: List[str] = []

    def fit(self, X: np.ndarray, feature_names: Optional[List[str]] = None) -> "BaseModalityTransformer":
        X = np.asarray(X, dtype=float)
        self._keep = np.isfinite(X).any(axis=0)
        if feature_names is not None:
            self.feature_names_ = [str(feature_names[i]) for i in np.flatnonzero(self._keep)]
        if self._keep.sum() == 0:
            return self
        Xk = X[:, self._keep]
        Xs = self._scaler.fit_transform(self._imp.fit_transform(Xk))
        n, d = Xs.shape
        n_comp = min(self.n_components, max(1, n - 1), d)
        self._pca = CPU_PCA(n_components=n_comp, random_state=self.random_state)
        self._pca.fit(Xs)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self._keep is None or self._keep.sum() == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        Xk = X[:, self._keep]
        Xs = self._scaler.transform(self._imp.transform(Xk))
        if self._pca is not None:
            Xs = self._pca.transform(Xs)
        return Xs.astype(np.float32)


class StandardiseObservedBlock:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.keep_ = None
        self.feature_names_: List[str] = []

    def fit(self, X: np.ndarray, feature_names: Optional[List[str]] = None) -> "StandardiseObservedBlock":
        X = np.asarray(X, dtype=float)
        self.keep_ = np.isfinite(X).any(axis=0)
        if feature_names is not None:
            self.feature_names_ = [str(feature_names[i]) for i in np.flatnonzero(self.keep_)]
        if self.keep_.sum() == 0:
            self.mean_ = np.zeros(0, dtype=np.float32)
            self.std_ = np.ones(0, dtype=np.float32)
            return self
        Xk = X[:, self.keep_]
        col_mean = np.nanmean(Xk, axis=0)
        col_std = np.nanstd(Xk, axis=0)
        col_std = np.where(col_std <= 1e-8, 1.0, col_std)
        self.mean_ = col_mean.astype(np.float32)
        self.std_ = col_std.astype(np.float32)
        return self

    def transform(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        X = np.asarray(X, dtype=float)
        if self.keep_ is None or self.keep_.sum() == 0:
            return (
                np.zeros((X.shape[0], 0), dtype=np.float32),
                np.zeros((X.shape[0], 0), dtype=bool),
            )
        Xk = X[:, self.keep_]
        miss = ~np.isfinite(Xk)
        Xfill = np.where(miss, self.mean_[None, :], Xk)
        Xstd = (Xfill - self.mean_[None, :]) / self.std_[None, :]
        return Xstd.astype(np.float32), miss

    def inverse_transform(self, X_std: np.ndarray) -> np.ndarray:
        X_std = np.asarray(X_std, dtype=np.float32)
        if X_std.shape[1] == 0:
            return X_std.astype(np.float32)
        return (X_std * self.std_[None, :] + self.mean_[None, :]).astype(np.float32)


class ProtJointPreprocessor:
    def __init__(self, max_features: int, min_obs_frac: float):
        self.max_features = int(max_features)
        self.min_obs_frac = float(min_obs_frac)
        self.scaler = StandardiseObservedBlock()
        self.selected_idx_: np.ndarray = np.zeros(0, dtype=int)
        self.feature_names_: List[str] = []

    def fit(self, X_train: np.ndarray, feature_names: List[str]) -> "ProtJointPreprocessor":
        X_train = np.asarray(X_train, dtype=float)
        obs_frac = np.isfinite(X_train).mean(axis=0)
        keep = obs_frac >= self.min_obs_frac
        if keep.sum() == 0:
            keep = np.isfinite(X_train).any(axis=0)
        idx = np.flatnonzero(keep)
        if idx.size == 0:
            self.selected_idx_ = np.zeros(0, dtype=int)
            self.feature_names_ = []
            self.scaler.fit(np.zeros((X_train.shape[0], 0)), feature_names=[])
            return self

        Xk = X_train[:, idx]
        var = np.nanvar(Xk, axis=0)
        order = np.argsort(-np.nan_to_num(var, nan=-1.0))
        idx = idx[order][: self.max_features]
        self.selected_idx_ = idx.astype(int)
        self.feature_names_ = [str(feature_names[i]) for i in self.selected_idx_]
        self.scaler.fit(X_train[:, self.selected_idx_], feature_names=self.feature_names_)
        self.feature_names_ = list(self.scaler.feature_names_)
        return self

    def transform(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        X = np.asarray(X, dtype=float)
        if self.selected_idx_.size == 0:
            return (
                np.zeros((X.shape[0], 0), dtype=np.float32),
                np.zeros((X.shape[0], 0), dtype=bool),
                [],
            )
        Xsel = X[:, self.selected_idx_]
        Xstd, miss = self.scaler.transform(Xsel)
        return Xstd, miss, list(self.feature_names_)

In [5]:
# Cache helpers

def _base_cache_path(seed: int, arm: str, fold_i: int) -> Path:
    return _cache_file(
        OUT_CACHE / "base",
        "base",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        fold_i=fold_i,
    )


def _prot_raw_cache_path(seed: int, arm: str, fold_i: int) -> Path:
    return _cache_file(
        OUT_CACHE / "prot_raw",
        "prot_raw",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        fold_i=fold_i,
    )


def _eval_cache_path(
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
) -> Path:
    return _cache_file(
        OUT_CACHE / "eval",
        "eval",
        cache_tag=CACHE_TAG,
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )


def load_or_build_base_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    train_cells: List[str],
    eligible_cells: List[str],
    rna_df: pd.DataFrame,
    cnv_df: pd.DataFrame,
    mut_df: pd.DataFrame,
) -> dict:
    path = _base_cache_path(seed, arm, fold_i)
    if path.exists():
        return load(path)

    rna_tr = BaseModalityTransformer(BASE_PCA_COMPONENTS, seed + 0).fit(
        rna_df.loc[train_cells].to_numpy(dtype=float),
        feature_names=rna_df.columns.astype(str).tolist(),
    )
    cnv_tr = BaseModalityTransformer(BASE_PCA_COMPONENTS, seed + 1).fit(
        cnv_df.loc[train_cells].to_numpy(dtype=float),
        feature_names=cnv_df.columns.astype(str).tolist(),
    )
    mut_tr = BaseModalityTransformer(BASE_PCA_COMPONENTS, seed + 2).fit(
        mut_df.loc[train_cells].to_numpy(dtype=float),
        feature_names=mut_df.columns.astype(str).tolist(),
    )

    payload = {
        "Xr": rna_tr.transform(rna_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xc": cnv_tr.transform(cnv_df.loc[eligible_cells].to_numpy(dtype=float)),
        "Xm": mut_tr.transform(mut_df.loc[eligible_cells].to_numpy(dtype=float)),
    }
    dump(payload, path)
    return payload


def load_or_build_prot_raw_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    train_cells: List[str],
    eligible_cells: List[str],
    prot_df: pd.DataFrame,
) -> dict:
    path = _prot_raw_cache_path(seed, arm, fold_i)
    if path.exists():
        return load(path)

    prot_proc = ProtJointPreprocessor(
        max_features=PROT_MAX_INPUT_FEATURES,
        min_obs_frac=PROT_MIN_TRAIN_OBS_FRAC,
    ).fit(
        prot_df.loc[train_cells].to_numpy(dtype=float),
        feature_names=prot_df.columns.astype(str).tolist(),
    )
    Xp, miss_p, prot_cols = prot_proc.transform(prot_df.loc[eligible_cells].to_numpy(dtype=float))
    payload = {
        "Xp": Xp,
        "Mp": (~miss_p).astype(np.float32),
        "miss_bool": miss_p,
        "prot_cols": prot_cols,
        "scaler": prot_proc.scaler,
        "selected_idx": prot_proc.selected_idx_,
    }
    dump(payload, path)
    return payload

In [6]:
# Data load
cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string").fillna("Unknown").astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print(f"\nCore cohort (RNA ∩ CNV ∩ MUT): {len(core_cells)} cell lines")
print(f"Group column for CV: {group_col}")

prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][["depmap_id", "compound_id", "y"]].copy()

prot_arm_data: Dict[str, pd.DataFrame] = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} not found at {p}, skipping.")

drug_cov = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print(f"\nSelected {len(selected_drugs)} drugs for task-aware joint bake-off by PRISM AUC coverage.")


# Missingness report
print("MISSINGNESS REPORT")

avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()

availability = pd.DataFrame(avail_rows, index=core_cells)
availability.index.name = "depmap_id"
(OUT_REPORTS / "modality_availability_matrix.csv").parent.mkdir(parents=True, exist_ok=True)
availability.to_csv(OUT_REPORTS / "modality_availability_matrix.csv")
print(f"Availability matrix: {OUT_REPORTS / 'modality_availability_matrix.csv'}  shape={availability.shape}")

avail_summary = (
    availability.sum().rename("n_cells_present").to_frame()
    .assign(
        n_cells_total=len(core_cells),
        pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2),
    )
)
avail_summary.to_csv(OUT_REPORTS / "modality_availability_summary.csv")
print("\nModality availability summary:")
display(avail_summary)

feat_miss_rows = []
for arm, df in prot_arm_data.items():
    df_core = df.reindex(core_cells)
    col_miss = df_core.isna().mean()
    row_miss = df_core.isna().mean(axis=1)
    feat_miss_rows.append({
        "arm": arm,
        "n_features": int(df_core.shape[1]),
        "n_cells_in_core": int(df_core.shape[0]),
        "overall_missing_pct": float(df_core.isna().mean().mean() * 100),
        "col_miss_q10": float(col_miss.quantile(0.10)),
        "col_miss_q25": float(col_miss.quantile(0.25)),
        "col_miss_q50": float(col_miss.quantile(0.50)),
        "col_miss_q75": float(col_miss.quantile(0.75)),
        "col_miss_q90": float(col_miss.quantile(0.90)),
        "col_miss_q99": float(col_miss.quantile(0.99)),
        "row_miss_q10": float(row_miss.quantile(0.10)),
        "row_miss_q50": float(row_miss.quantile(0.50)),
        "row_miss_q90": float(row_miss.quantile(0.90)),
        "n_cols_fully_missing": int((col_miss == 1.0).sum()),
        "n_rows_fully_missing": int((row_miss == 1.0).sum()),
        "n_cols_zero_missing": int((col_miss == 0.0).sum()),
    })
feat_miss_df = pd.DataFrame(feat_miss_rows)
feat_miss_df.to_csv(OUT_REPORTS / "per_arm_feature_missingness.csv", index=False)
print("\nPer-arm feature missingness:")
display(feat_miss_df)

pat_counts = None
platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0

    def pattern_label(row) -> str:
        parts = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(parts) if parts else "none"

    pat_counts = (
        plat_pres.apply(pattern_label, axis=1).rename("pattern")
        .value_counts().rename_axis("pattern").reset_index(name="n_cells")
    )
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    print("\nCombined union platform patterns:")
    display(pat_counts)

    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols:
            continue
        n_absent = int(plat_pres[key].eq(0).sum())
        miss_from_abs = n_absent * len(cols)
        miss_total = int(union_df[cols].isna().sum().sum())
        pm_rows.append({
            "platform": key,
            "n_features": len(cols),
            "n_cells_present": int(plat_pres[key].sum()),
            "n_cells_absent": n_absent,
            "frac_cells_present": float(plat_pres[key].mean()),
            "pct_missingness_from_platform_absence": (
                float(miss_from_abs / miss_total * 100) if miss_total > 0 else np.nan
            ),
        })
    platform_miss_df = pd.DataFrame(pm_rows)
    platform_miss_df.to_csv(OUT_REPORTS / "combined_union_platform_missingness_contrib.csv", index=False)
    print("\nCombined union, missingness from platform absence:")
    display(platform_miss_df)


def build_missingness_report(
    avail_summary: pd.DataFrame,
    feat_miss_df: pd.DataFrame,
    pat_counts: Optional[pd.DataFrame],
    platform_miss_df: Optional[pd.DataFrame],
    track2_arm: str,
    all_seeds: List[int],
    ts: str,
) -> dict:
    report = {
        "generated_at": ts,
        "seeds": all_seeds,
        "active_arms": ACTIVE_ARMS,
        "deprioritised_arm": DEPRIORITISED_ARM,
        "leakage_note": (
            "All raw missingness summaries are computed before any imputation. "
            "All modelling, imputation, and prediction later remain fold-safe."
        ),
        "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
        "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
        "taskaware_settings": {
            "base_pca_components": BASE_PCA_COMPONENTS,
            "prot_max_input_features": PROT_MAX_INPUT_FEATURES,
            "joint_latent_dim": JOINT_LATENT_DIM,
            "joint_hidden_dim": JOINT_HIDDEN_DIM,
            "joint_epochs": JOINT_EPOCHS,
            "joint_patience": JOINT_PATIENCE,
            "joint_batch_size": JOINT_BATCH_SIZE,
            "joint_remask_frac": JOINT_REMASK_FRAC,
        },
    }
    if pat_counts is not None:
        report["combined_union_platform_patterns"] = {
            "arm": track2_arm,
            "structural_missingness_warning": (
                "Combined-union missingness is heavily shaped by platform availability. "
                "Indicator-aware strategies remain important for this arm."
            ),
            "patterns": pat_counts.to_dict(orient="records"),
        }
    if platform_miss_df is not None:
        report["combined_union_platform_missingness_contrib"] = platform_miss_df.to_dict(orient="records")
    report["bakeoff_outputs"] = {
        "detail_file": f"taskaware_bakeoff_merged_{len(all_seeds)}seeds.csv",
        "summary_file": f"taskaware_bakeoff_summary_merged_{len(all_seeds)}seeds.csv",
        "lock_file": TASKAWARE_LOCK_FILENAME,
    }
    return report


missingness_report = build_missingness_report(
    avail_summary=avail_summary,
    feat_miss_df=feat_miss_df,
    pat_counts=pat_counts,
    platform_miss_df=platform_miss_df,
    track2_arm=TRACK2_ARM,
    all_seeds=ALL_SEEDS,
    ts=datetime.now(timezone.utc).isoformat(),
)
write_json(missingness_report, OUT_REPORTS / "missingness_report.json")
print(f"\nMissingness report written: {OUT_REPORTS / 'missingness_report.json'}")


Core cohort (RNA ∩ CNV ∩ MUT): 1079 cell lines
Group column for CV: lineage_1
Loaded prot_combined_union: (1079, 18751)
Loaded prot_ms_ccle_gygi: (1079, 11780)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)

Selected 100 drugs for task-aware joint bake-off by PRISM AUC coverage.
MISSINGNESS REPORT
Availability matrix: artifacts/reports/notebook 8_taskaware_joint/modality_availability_matrix.csv  shape=(1079, 7)

Modality availability summary:


,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_ms_ccle_gygi,304,1079,28.17
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72



Per-arm feature missingness:


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
1,prot_ms_ccle_gygi,11780,1079,78.876396,0.718258,0.718258,0.732159,0.861214,0.949027,0.979611,0.237267,1.000000,1.0,0,775,0
2,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
3,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0



Combined union platform patterns:


,pattern,n_cells,frac_cells
0,none,400,0.370714
1,ms+rppa+procan,233,0.215941
2,rppa+procan,187,0.173309
3,rppa,128,0.118628
4,ms+rppa,64,0.059314
5,procan,60,0.055607
6,ms+procan,5,0.004634
7,ms,2,0.001854



Combined union, missingness from platform absence:


,platform,n_features,n_cells_present,n_cells_absent,frac_cells_present,pct_missingness_from_platform_absence
0,ms,10847,304,775,0.281742,92.892390
1,rppa,144,612,467,0.567192,100.000000
2,procan,7760,485,594,0.449490,78.405864



Missingness report written: artifacts/reports/notebook 8_taskaware_joint/missingness_report.json


In [7]:
# Task-aware joint model
class MLPEncoder(nn.Module):
    def __init__(self, in_dim: int, latent_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.in_dim = int(in_dim)
        if self.in_dim > 0:
            self.net = nn.Sequential(
                nn.Linear(self.in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, latent_dim),
                nn.LayerNorm(latent_dim),
            )
        else:
            self.net = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.net is None:
            return torch.zeros((x.shape[0], 0), device=x.device, dtype=x.dtype)
        return self.net(x)


class TaskAwareJointModel(nn.Module):
    def __init__(
        self,
        d_rna: int,
        d_cnv: int,
        d_mut: int,
        d_prot: int,
        latent_dim: int,
        hidden_dim: int,
        dropout: float,
        add_indicators: bool,
        use_modalities: Tuple[str, ...],
    ):
        super().__init__()
        self.use_modalities = tuple(use_modalities)
        self.use_prot = "prot" in self.use_modalities and int(d_prot) > 0
        self.add_indicators = bool(add_indicators)
        self.latent_dim = int(latent_dim)
        self.d_prot = int(d_prot)

        self.enc_rna = MLPEncoder(d_rna if "rna" in self.use_modalities else 0, latent_dim, hidden_dim, dropout)
        self.enc_cnv = MLPEncoder(d_cnv if "cnv" in self.use_modalities else 0, latent_dim, hidden_dim, dropout)
        self.enc_mut = MLPEncoder(d_mut if "mut" in self.use_modalities else 0, latent_dim, hidden_dim, dropout)

        prot_in_dim = self.d_prot * (2 if self.add_indicators else 1) if self.use_prot else 0
        self.enc_prot = MLPEncoder(prot_in_dim, latent_dim, hidden_dim, dropout)

        self.gate_net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )
        self.pred_head = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

        if self.use_prot:
            self.prot_decoder = nn.Sequential(
                nn.Linear(latent_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, self.d_prot),
            )
        else:
            self.prot_decoder = None

    def _fuse(self, latents: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        names = [k for k, v in latents.items() if v is not None and v.shape[1] > 0]
        if len(names) == 0:
            raise ValueError("No active modalities were provided to the fusion block.")
        if len(names) == 1:
            fused = latents[names[0]]
            return fused, {names[0]: torch.ones((fused.shape[0],), device=fused.device, dtype=fused.dtype)}

        stack = torch.stack([latents[n] for n in names], dim=1)
        scores = self.gate_net(stack).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        fused = (weights.unsqueeze(-1) * stack).sum(dim=1)
        return fused, {n: weights[:, i] for i, n in enumerate(names)}

    def forward(
        self,
        xr: torch.Tensor,
        xc: torch.Tensor,
        xm: torch.Tensor,
        xp: torch.Tensor,
        mp: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        latents: Dict[str, torch.Tensor] = {}
        if "rna" in self.use_modalities and xr.shape[1] > 0:
            latents["rna"] = self.enc_rna(xr)
        if "cnv" in self.use_modalities and xc.shape[1] > 0:
            latents["cnv"] = self.enc_cnv(xc)
        if "mut" in self.use_modalities and xm.shape[1] > 0:
            latents["mut"] = self.enc_mut(xm)
        prot_latent = None
        if self.use_prot and xp.shape[1] > 0:
            prot_in = torch.cat([xp, mp], dim=1) if self.add_indicators else xp
            prot_latent = self.enc_prot(prot_in)
            latents["prot"] = prot_latent

        fused, gate_weights = self._fuse(latents)
        yhat = self.pred_head(fused).squeeze(-1)

        recon = None
        if self.use_prot and prot_latent is not None and self.prot_decoder is not None:
            recon = self.prot_decoder(torch.cat([prot_latent, fused], dim=1))

        return {
            "yhat": yhat,
            "recon": recon,
            "fused": fused,
            "gate_weights": gate_weights,
            "latents": latents,
        }


class TaskAwareTrainer:
    def __init__(
        self,
        model: TaskAwareJointModel,
        pred_weight: float,
        recon_weight: float,
        remask_frac: float,
        lr: float,
        weight_decay: float,
        epochs: int,
        patience: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model = model.to(device)
        self.pred_weight = float(pred_weight)
        self.recon_weight = float(recon_weight)
        self.remask_frac = float(remask_frac)
        self.lr = float(lr)
        self.weight_decay = float(weight_decay)
        self.epochs = int(epochs)
        self.patience = int(patience)
        self.batch_size = int(batch_size)
        self.device = device

    def _make_loader(
        self,
        xr: np.ndarray,
        xc: np.ndarray,
        xm: np.ndarray,
        xp: np.ndarray,
        mp: np.ndarray,
        y: np.ndarray,
        shuffle: bool,
    ) -> DataLoader:
        ds = TensorDataset(
            torch.from_numpy(xr.astype(np.float32)),
            torch.from_numpy(xc.astype(np.float32)),
            torch.from_numpy(xm.astype(np.float32)),
            torch.from_numpy(xp.astype(np.float32)),
            torch.from_numpy(mp.astype(np.float32)),
            torch.from_numpy(y.astype(np.float32)),
        )
        return DataLoader(ds, batch_size=min(self.batch_size, len(ds)), shuffle=shuffle, drop_last=False)

    def _loss_components(
        self,
        out: Dict[str, torch.Tensor],
        y_true: torch.Tensor,
        xp_target: torch.Tensor,
        mp_full: torch.Tensor,
        remask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        pred_loss = F.mse_loss(out["yhat"], y_true)
        recon_loss = torch.tensor(0.0, device=y_true.device)
        if out["recon"] is not None and xp_target.shape[1] > 0:
            if remask is not None and remask.float().sum() > 0:
                loss_mask = remask
            else:
                loss_mask = mp_full > 0.5
            denom = loss_mask.float().sum().clamp_min(1.0)
            recon_loss = (((out["recon"] - xp_target) ** 2) * loss_mask.float()).sum() / denom
        total = self.pred_weight * pred_loss + self.recon_weight * recon_loss
        return total, pred_loss, recon_loss

    def fit(
        self,
        xr_train: np.ndarray,
        xc_train: np.ndarray,
        xm_train: np.ndarray,
        xp_train: np.ndarray,
        mp_train: np.ndarray,
        y_train: np.ndarray,
        seed: int,
    ) -> Dict[str, float]:
        tr_idx, val_idx = _split_train_val_idx(len(y_train), seed)
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        best_state = copy.deepcopy(self.model.state_dict())
        best_val = math.inf
        bad_epochs = 0
        rng = np.random.default_rng(seed)

        train_loader = self._make_loader(
            xr=xr_train[tr_idx],
            xc=xc_train[tr_idx],
            xm=xm_train[tr_idx],
            xp=xp_train[tr_idx],
            mp=mp_train[tr_idx],
            y=y_train[tr_idx],
            shuffle=True,
        )

        if len(val_idx) > 0:
            val_tensors = {
                "xr": torch.from_numpy(xr_train[val_idx].astype(np.float32)).to(self.device),
                "xc": torch.from_numpy(xc_train[val_idx].astype(np.float32)).to(self.device),
                "xm": torch.from_numpy(xm_train[val_idx].astype(np.float32)).to(self.device),
                "xp": torch.from_numpy(xp_train[val_idx].astype(np.float32)).to(self.device),
                "mp": torch.from_numpy(mp_train[val_idx].astype(np.float32)).to(self.device),
                "y": torch.from_numpy(y_train[val_idx].astype(np.float32)).to(self.device),
            }
        else:
            val_tensors = None

        for _epoch in range(self.epochs):
            self.model.train()
            for xb_r, xb_c, xb_m, xb_p, mb_p, yb in train_loader:
                xb_r = xb_r.to(self.device)
                xb_c = xb_c.to(self.device)
                xb_m = xb_m.to(self.device)
                xb_p = xb_p.to(self.device)
                mb_p = mb_p.to(self.device)
                yb = yb.to(self.device)

                remask_np = build_random_remask(mb_p.detach().cpu().numpy() > 0.5, self.remask_frac, rng)
                remask = torch.from_numpy(remask_np.astype(np.float32)).to(self.device)
                mp_in = mb_p * (1.0 - remask)
                xp_in = xb_p * mp_in

                out = self.model(xr=xb_r, xc=xb_c, xm=xb_m, xp=xp_in, mp=mp_in)
                total, _, _ = self._loss_components(out, yb, xb_p, mb_p, remask=remask > 0.5)

                opt.zero_grad()
                total.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                opt.step()

            self.model.eval()
            with torch.no_grad():
                if val_tensors is not None:
                    out_val = self.model(
                        xr=val_tensors["xr"],
                        xc=val_tensors["xc"],
                        xm=val_tensors["xm"],
                        xp=val_tensors["xp"] * val_tensors["mp"],
                        mp=val_tensors["mp"],
                    )
                    val_total, _, _ = self._loss_components(
                        out=out_val,
                        y_true=val_tensors["y"],
                        xp_target=val_tensors["xp"],
                        mp_full=val_tensors["mp"],
                        remask=None,
                    )
                    current = float(val_total.detach().cpu().item())
                else:
                    current = 0.0

            if current < (best_val - 1e-6):
                best_val = current
                best_state = copy.deepcopy(self.model.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    break

        self.model.load_state_dict(best_state)
        self.model.eval()
        return {"best_val_total": float(best_val)}

    @torch.no_grad()
    def predict(
        self,
        xr: np.ndarray,
        xc: np.ndarray,
        xm: np.ndarray,
        xp: np.ndarray,
        mp: np.ndarray,
    ) -> Dict[str, np.ndarray]:
        xr_t = torch.from_numpy(xr.astype(np.float32)).to(self.device)
        xc_t = torch.from_numpy(xc.astype(np.float32)).to(self.device)
        xm_t = torch.from_numpy(xm.astype(np.float32)).to(self.device)
        xp_t = torch.from_numpy(xp.astype(np.float32)).to(self.device)
        mp_t = torch.from_numpy(mp.astype(np.float32)).to(self.device)
        out = self.model(xr=xr_t, xc=xc_t, xm=xm_t, xp=xp_t * mp_t, mp=mp_t)
        gate_np = {k: v.detach().cpu().numpy() for k, v in out["gate_weights"].items()}
        return {
            "pred": out["yhat"].detach().cpu().numpy().astype(np.float32),
            "recon": None if out["recon"] is None else out["recon"].detach().cpu().numpy().astype(np.float32),
            "fused": out["fused"].detach().cpu().numpy().astype(np.float32),
            "gates": gate_np,
        }


# Joint evaluation helpers

def zeros_block(n: int) -> np.ndarray:
    return np.zeros((n, 0), dtype=np.float32)


def build_modality_arrays(
    base_mats: Dict[str, np.ndarray],
    prot_payload: Dict[str, object],
    row_idx: np.ndarray,
    feat_keys: Tuple[str, ...],
) -> Dict[str, np.ndarray]:
    out = {
        "xr": base_mats["rna"][row_idx] if "rna" in feat_keys else zeros_block(len(row_idx)),
        "xc": base_mats["cnv"][row_idx] if "cnv" in feat_keys else zeros_block(len(row_idx)),
        "xm": base_mats["mut"][row_idx] if "mut" in feat_keys else zeros_block(len(row_idx)),
        "xp": prot_payload["Xp"][row_idx] if "prot" in feat_keys else zeros_block(len(row_idx)),
        "mp": prot_payload["Mp"][row_idx] if "prot" in feat_keys else zeros_block(len(row_idx)),
    }
    return out


def fit_and_eval_joint(
    seed: int,
    feature_set: str,
    strategy_name: str,
    add_indicators: bool,
    train_arrays: Dict[str, np.ndarray],
    test_arrays: Dict[str, np.ndarray],
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> Dict[str, object]:
    feat_keys = parse_feature_set(feature_set)
    model = TaskAwareJointModel(
        d_rna=train_arrays["xr"].shape[1],
        d_cnv=train_arrays["xc"].shape[1],
        d_mut=train_arrays["xm"].shape[1],
        d_prot=train_arrays["xp"].shape[1],
        latent_dim=JOINT_LATENT_DIM,
        hidden_dim=JOINT_HIDDEN_DIM,
        dropout=JOINT_DROPOUT,
        add_indicators=add_indicators,
        use_modalities=feat_keys,
    )
    trainer = TaskAwareTrainer(
        model=model,
        pred_weight=JOINT_PRED_LOSS_WEIGHT,
        recon_weight=JOINT_RECON_LOSS_WEIGHT if "prot" in feat_keys else 0.0,
        remask_frac=JOINT_REMASK_FRAC if "prot" in feat_keys else 0.0,
        lr=JOINT_LR,
        weight_decay=JOINT_WEIGHT_DECAY,
        epochs=JOINT_EPOCHS,
        patience=JOINT_PATIENCE,
        batch_size=JOINT_BATCH_SIZE,
        device=TORCH_DEVICE,
    )
    fit_meta = trainer.fit(
        xr_train=train_arrays["xr"],
        xc_train=train_arrays["xc"],
        xm_train=train_arrays["xm"],
        xp_train=train_arrays["xp"],
        mp_train=train_arrays["mp"],
        y_train=y_train,
        seed=seed,
    )
    pred_out = trainer.predict(
        xr=test_arrays["xr"],
        xc=test_arrays["xc"],
        xm=test_arrays["xm"],
        xp=test_arrays["xp"],
        mp=test_arrays["mp"],
    )
    recon_mse = np.nan
    if pred_out["recon"] is not None and test_arrays["xp"].shape[1] > 0:
        mask = test_arrays["mp"] > 0.5
        if mask.sum() > 0:
            recon_mse = float((((pred_out["recon"] - test_arrays["xp"]) ** 2) * mask).sum() / mask.sum())
    gate_means = {f"gate_{k}_mean": float(np.mean(v)) for k, v in pred_out["gates"].items()}
    return {
        "pred": pred_out["pred"],
        "recon_mse_obs": recon_mse,
        "best_val_total": float(fit_meta.get("best_val_total", np.nan)),
        **gate_means,
    }


def load_or_run_eval_cache(
    *,
    seed: int,
    arm: str,
    drug: str,
    fold_i: int,
    cfg_rank: int,
    model_name: str,
    feature_set: str,
    imputer_strategy: str,
    extra_meta: dict,
    train_arrays: Dict[str, np.ndarray],
    test_arrays: Dict[str, np.ndarray],
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> dict:
    path = _eval_cache_path(
        seed=seed,
        arm=arm,
        drug=drug,
        fold_i=fold_i,
        cfg_rank=cfg_rank,
        model_name=model_name,
        feature_set=feature_set,
        imputer_strategy=imputer_strategy,
    )
    if path.exists():
        return load(path)

    add_indicators = bool(extra_meta.get("add_indicators", False))
    eval_out = fit_and_eval_joint(
        seed=seed,
        feature_set=feature_set,
        strategy_name=imputer_strategy,
        add_indicators=add_indicators,
        train_arrays=train_arrays,
        test_arrays=test_arrays,
        y_train=y_train,
        y_test=y_test,
    )

    row = {
        "seed": seed,
        "config_rank": cfg_rank,
        "arm": arm,
        "model": model_name,
        "feature_set": feature_set,
        "compound_id": drug,
        "fold": fold_i,
        "imputer_strategy": imputer_strategy,
        "n_train": int(len(y_train)),
        "n_test": int(len(y_test)),
        "spearman": spearman_corr(y_test, eval_out["pred"]),
        "r2": float(r2_score(y_test, eval_out["pred"])),
        **extra_meta,
        **{k: v for k, v in eval_out.items() if k != "pred"},
    }
    dump(row, path)
    return row


# Task-aware proteomics bake-off
print("TASK-AWARE JOINT IMPUTATION + PREDICTION BAKE-OFF (3 seeds, leakage-safe)")

all_bakeoff_rows: List[dict] = []
seeds_to_run: List[int] = []
REQUIRED_NEW_COLS = {"config_rank", "arm", "model", "feature_set", "uses_prot"}

for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"taskaware_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)
        if (
            (not REQUIRED_NEW_COLS.issubset(existing.columns))
            or (not checkpoint_matches_expected_grid(existing, EXPECTED_CONFIG_KEYS))
        ):
            print(f"[resume] Seed {run_seed} file exists but is from an old schema/grid, rerunning.")
            seeds_to_run.append(run_seed)
            continue
        existing["seed"] = existing["seed"].astype(int)
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping task-aware loop.")
else:
    print(f"[resume] Seeds remaining: {seeds_to_run}")

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded.")
            continue

        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            print(f"  [SKIP] {arm} has no configs assigned.")
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_ckpt_path = OUT_REPORTS / f"taskaware_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs: set = set()

        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)
            if (
                REQUIRED_NEW_COLS.issubset(arm_existing.columns)
                and checkpoint_matches_expected_grid(arm_existing, EXPECTED_CONFIG_KEYS_BY_ARM[arm])
            ):
                arm_existing["seed"] = arm_existing["seed"].astype(int)
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    already_done_drugs = set(arm_existing["compound_id"].unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema/grid, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed)
        print(f"  {arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        fold_cache: Dict[int, dict] = {}
        print(f"    Building/loading fold caches for {arm}...")
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]
            base_payload = load_or_build_base_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
                rna_df=rna,
                cnv_df=cnv,
                mut_df=mut,
            )
            prot_payload = load_or_build_prot_raw_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
                prot_df=prot_core.loc[eligible_cells],
            )
            fold_cache[fold_i] = {
                "base_mats": {
                    "rna": base_payload["Xr"],
                    "cnv": base_payload["Xc"],
                    "mut": base_payload["Xm"],
                },
                "prot_payload": prot_payload,
            }

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            if drug in already_done_drugs:
                continue

            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1
            if n_run % 5 == 0:
                arm_rows_partial = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
                if arm_rows_partial:
                    arm_df_partial = pd.DataFrame(arm_rows_partial)
                    arm_df_partial.to_csv(arm_ckpt_path, index=False)
                    cumulative_done = arm_df_partial["compound_id"].nunique()
                    print(f"    [mid-checkpoint] drug {cumulative_done}/{N_DRUGS_BAKEOFF} | {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                base_mats = fold_cache[fold_i]["base_mats"]
                prot_payload = fold_cache[fold_i]["prot_payload"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    uses_prot = "prot" in cfg_keys

                    if not uses_prot:
                        train_arrays = build_modality_arrays(base_mats, prot_payload, idx_train, cfg_keys)
                        test_arrays = build_modality_arrays(base_mats, prot_payload, idx_test, cfg_keys)
                        if sum(arr.shape[1] for arr in [train_arrays["xr"], train_arrays["xc"], train_arrays["xm"]]) == 0:
                            continue
                        row = load_or_run_eval_cache(
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            imputer_strategy="no_prot_reference",
                            extra_meta={
                                "uses_prot": False,
                                "add_indicators": False,
                                "force_indicators": False,
                            },
                            train_arrays=train_arrays,
                            test_arrays=test_arrays,
                            y_train=y_train,
                            y_test=y_test,
                        )
                        all_bakeoff_rows.append(row)
                        continue

                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    if len(cfg_keys_wo_prot) > 0:
                        train_ref = build_modality_arrays(base_mats, prot_payload, idx_train, cfg_keys_wo_prot)
                        test_ref = build_modality_arrays(base_mats, prot_payload, idx_test, cfg_keys_wo_prot)
                        if sum(arr.shape[1] for arr in [train_ref["xr"], train_ref["xc"], train_ref["xm"]]) > 0:
                            row = load_or_run_eval_cache(
                                seed=run_seed,
                                arm=arm,
                                drug=drug,
                                fold_i=fold_i,
                                cfg_rank=cfg_rank,
                                model_name=cfg_model,
                                feature_set=cfg_feature_set,
                                imputer_strategy="reference_without_prot",
                                extra_meta={
                                    "uses_prot": True,
                                    "add_indicators": False,
                                    "force_indicators": False,
                                },
                                train_arrays=train_ref,
                                test_arrays=test_ref,
                                y_train=y_train,
                                y_test=y_test,
                            )
                            all_bakeoff_rows.append(row)

                    for strat in TASKAWARE_STRATEGIES:
                        strat_name = str(strat["name"])
                        add_ind = bool(strat["add_indicators"])
                        train_arrays = build_modality_arrays(base_mats, prot_payload, idx_train, cfg_keys)
                        test_arrays = build_modality_arrays(base_mats, prot_payload, idx_test, cfg_keys)
                        if train_arrays["xp"].shape[1] == 0:
                            continue
                        row = load_or_run_eval_cache(
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            imputer_strategy=strat_name,
                            extra_meta={
                                "uses_prot": True,
                                "add_indicators": add_ind,
                                "force_indicators": (arm == TRACK2_ARM),
                            },
                            train_arrays=train_arrays,
                            test_arrays=test_arrays,
                            y_train=y_train,
                            y_test=y_test,
                        )
                        all_bakeoff_rows.append(row)

        print(f"    drugs_run={n_run}, drugs_skipped={n_skip}")
        arm_rows = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
        arm_df = pd.DataFrame(arm_rows)
        arm_df.to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}  shape={arm_df.shape}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"taskaware_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")

TASK-AWARE JOINT IMPUTATION + PREDICTION BAKE-OFF (3 seeds, leakage-safe)
[resume] Seeds remaining: [19537, 1584678, 17052356]
  Seed 19537
  prot_combined_union: GroupKFold(n_splits=10), 679 cells
    Building/loading fold caches for prot_combined_union...
    [mid-checkpoint] drug 4/100 | artifacts/reports/notebook 8_taskaware_joint/taskaware_bakeoff_seed19537_prot_combined_union.csv


KeyboardInterrupt: 

In [ ]:
# Summaries
bakeoff_df = pd.DataFrame(all_bakeoff_rows)
merged_path = OUT_REPORTS / f"taskaware_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print(f"\nMerged task-aware bake-off: {merged_path}  shape={bakeoff_df.shape}")

if bakeoff_df.shape[0] == 0:
    raise RuntimeError("Task-aware bake-off produced no rows. Check arm availability and cell thresholds.")

drug_means = (
    bakeoff_df
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy", "compound_id"],
        as_index=False,
    )
    .agg(
        spearman=("spearman", safe_nanmean),
        r2=("r2", safe_nanmean),
        recon_mse_obs=("recon_mse_obs", safe_nanmean),
        n_folds=("fold", "nunique"),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(
        ["config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
        mean_recon_mse_obs=("recon_mse_obs", safe_nanmean),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

base_ref = (
    bakeoff_summary[
        bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ][["config_rank", "arm", "model", "feature_set", "mean_spearman", "imputer_strategy"]]
    .rename(columns={"mean_spearman": "baseline_mean_spearman", "imputer_strategy": "baseline_strategy"})
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

bakeoff_summary = bakeoff_summary.merge(
    base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)
bakeoff_summary["delta_vs_baseline"] = np.where(
    bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"]),
    0.0,
    bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"],
)
summary_path = OUT_REPORTS / f"taskaware_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
print("\nTask-aware bake-off summary:")
display(bakeoff_summary)

per_seed_summary = (
    drug_means
    .groupby(
        ["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"],
        as_index=False,
    )
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
    )
    .sort_values(["config_rank", "seed", "mean_spearman"], ascending=[True, True, False])
)
per_seed_path = OUT_REPORTS / "taskaware_bakeoff_per_seed_summary.csv"
per_seed_summary.to_csv(per_seed_path, index=False)
print("\nPer-seed task-aware summary:")
display(per_seed_summary)


# Lock decision
print("TASK-AWARE JOINT LOCK DECISION")

taskaware_choice = {}
for cfg in TASKAWARE_TEST_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    cfg_uses_prot = "prot" in parse_feature_set(cfg_feature_set)
    key = f"rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = bakeoff_summary[
        (bakeoff_summary["config_rank"] == cfg_rank) &
        (bakeoff_summary["arm"] == cfg_arm) &
        (bakeoff_summary["model"] == cfg_model) &
        (bakeoff_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        taskaware_choice[key] = {"chosen_strategy": None, "reason": "No task-aware bake-off rows produced for this config."}
        continue

    if not cfg_uses_prot:
        ref_rows = cfg_df[cfg_df["imputer_strategy"] == "no_prot_reference"]
        ref_sp = float(ref_rows["mean_spearman"].iloc[0]) if len(ref_rows) else np.nan
        taskaware_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": "not_applicable",
            "mean_spearman_chosen": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "mean_spearman_baseline": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "delta_vs_baseline": 0.0,
            "n_seeds_in_estimate": len(ALL_SEEDS),
            "reason": "This ranked configuration does not contain proteomics, so task-aware joint proteomics imputation is not applicable.",
        }
        print(f"\n{key}:")
        print("  Chosen   : not_applicable")
        continue

    candidates = cfg_df[~cfg_df["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])].copy()
    if cfg_arm == TRACK2_ARM:
        ind_candidates = candidates[candidates["imputer_strategy"].str.contains("indicator", regex=False)]
        if ind_candidates.shape[0] > 0:
            candidates = ind_candidates

    if candidates.shape[0] == 0:
        taskaware_choice[key] = {"chosen_strategy": None, "reason": "No task-aware candidates available for this config."}
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_prot"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan
    chosen = str(best["imputer_strategy"])
    mean_sp = float(best["mean_spearman"])
    std_sp = float(best["std_spearman"])
    delta = float(best["delta_vs_baseline"]) if pd.notna(best["delta_vs_baseline"]) else np.nan

    taskaware_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": chosen,
        "mean_spearman_chosen": round(mean_sp, 6),
        "std_spearman_chosen": round(std_sp, 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(delta, 6) if np.isfinite(delta) else None,
        "n_seeds_in_estimate": len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across {len(ALL_SEEDS)} seeds"
            + (
                f"; delta vs config-specific no-prot reference: {delta:+.4f}."
                if np.isfinite(delta) else
                "; no config-specific no-prot reference available."
            )
            + (" Indicators mandatory for the combined-union arm." if cfg_arm == TRACK2_ARM else "")
        ),
    }
    print(f"\n{key}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}")
    if np.isfinite(base_sp):
        print(f"  Baseline : {base_sp:.4f}  delta={delta:+.4f}")

taskaware_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "base_pca_components": BASE_PCA_COMPONENTS,
    "prot_max_input_features": PROT_MAX_INPUT_FEATURES,
    "joint_settings": {
        "latent_dim": JOINT_LATENT_DIM,
        "hidden_dim": JOINT_HIDDEN_DIM,
        "dropout": JOINT_DROPOUT,
        "epochs": JOINT_EPOCHS,
        "patience": JOINT_PATIENCE,
        "batch_size": JOINT_BATCH_SIZE,
        "lr": JOINT_LR,
        "weight_decay": JOINT_WEIGHT_DECAY,
        "pred_loss_weight": JOINT_PRED_LOSS_WEIGHT,
        "recon_loss_weight": JOINT_RECON_LOSS_WEIGHT,
        "remask_frac": JOINT_REMASK_FRAC,
    },
    "test_configs": TASKAWARE_TEST_CONFIGS,
    "note": (
        "This notebook trains a task-aware joint neural model that predicts drug response while simultaneously "
        "reconstructing proteomics from masked inputs. RNA, CNV, and mutation are encoded as fold-safe PCA "
        "embeddings, proteomics is encoded from a fold-safe raw block with optional missingness indicators, "
        "and fusion uses modality attention weights. All preprocessing, training, and evaluation are strictly fold-safe."
    ),
    "config_choices": taskaware_choice,
}

taskaware_choice_path = OUT_META / TASKAWARE_LOCK_FILENAME
write_json(taskaware_choice_doc, taskaware_choice_path)
print(f"\nTask-aware joint choice written: {taskaware_choice_path}")

global_copy = IN_META / TASKAWARE_LOCK_FILENAME
write_json(taskaware_choice_doc, global_copy)
print(f"Global copy written: {global_copy}")


# Interpretability helpers

def choose_interpretability_target(
    detail_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    arm: Optional[str] = None,
    model: Optional[str] = None,
    feature_set: Optional[str] = None,
    seed: Optional[int] = None,
) -> Tuple[int, str, str, str, str, str]:
    cand = summary_df.copy()
    cand = cand[~cand["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])]
    cand = cand[cand["uses_prot"] == True]
    if arm is not None:
        cand = cand[cand["arm"] == arm]
    if model is not None:
        cand = cand[cand["model"] == model]
    if feature_set is not None:
        cand = cand[cand["feature_set"] == feature_set]
    if cand.empty:
        raise ValueError("No candidate summary rows found for interpretability target.")
    best_summary = cand.sort_values(["mean_spearman", "mean_r2"], ascending=[False, False]).iloc[0]

    chosen_seed = int(seed if seed is not None else ALL_SEEDS[0])
    chosen_arm = str(best_summary["arm"])
    chosen_model = str(best_summary["model"])
    chosen_fs = str(best_summary["feature_set"])
    chosen_strategy = str(best_summary["imputer_strategy"])

    detail_rows = detail_df[
        (detail_df["seed"] == chosen_seed) &
        (detail_df["arm"] == chosen_arm) &
        (detail_df["model"] == chosen_model) &
        (detail_df["feature_set"] == chosen_fs) &
        (detail_df["imputer_strategy"] == chosen_strategy)
    ].copy()
    if detail_rows.empty:
        raise ValueError("No detail rows found for interpretability target.")
    drug_rank = detail_rows.groupby("compound_id", as_index=False)["spearman"].mean().sort_values("spearman", ascending=False)
    chosen_drug = str(drug_rank.iloc[0]["compound_id"])
    return chosen_seed, chosen_arm, chosen_model, chosen_fs, chosen_strategy, chosen_drug


def build_original_block_filtered(
    df: pd.DataFrame,
    train_cells: List[str],
    all_cells: List[str],
    top_k: int,
    strategy: str = "median",
    do_scale: bool = True,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    X_train = df.loc[train_cells].to_numpy(dtype=float)
    X_all = df.loc[all_cells].to_numpy(dtype=float)
    keep = np.isfinite(X_train).any(axis=0)
    if keep.sum() == 0:
        return zeros_block(len(train_cells)), zeros_block(len(all_cells)), []

    cols = df.columns[keep].astype(str).tolist()
    X_train = X_train[:, keep]
    X_all = X_all[:, keep]
    var = np.nanvar(X_train, axis=0)
    order = np.argsort(-np.nan_to_num(var, nan=-1.0))[:top_k]
    X_train = X_train[:, order]
    X_all = X_all[:, order]
    cols = [cols[i] for i in order]

    imp = SimpleImputer(strategy=strategy)
    X_train_imp = imp.fit_transform(X_train)
    X_all_imp = imp.transform(X_all)
    if do_scale:
        sc = StandardScaler()
        X_train_imp = sc.fit_transform(X_train_imp)
        X_all_imp = sc.transform(X_all_imp)
    return X_train_imp.astype(np.float32), X_all_imp.astype(np.float32), cols


def train_case_model(
    seed: int,
    arm: str,
    feature_set: str,
    strategy_name: str,
    fold_i: int,
    drug: str,
):
    prot_df = prot_arm_data[arm].copy()
    prot_df.index = prot_df.index.astype(str).str.strip()
    prot_core = prot_df.reindex(core_cells)
    eligible_cells = sorted(prot_core.notna().any(axis=1)[lambda s: s].index.tolist())
    arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
    splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=seed)
    train_idx, test_idx = splits[fold_i]
    train_cells = [eligible_cells[int(i)] for i in train_idx]
    test_cells = [eligible_cells[int(i)] for i in test_idx]

    base_payload = load_or_build_base_fold_cache(
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        train_cells=train_cells,
        eligible_cells=eligible_cells,
        rna_df=rna,
        cnv_df=cnv,
        mut_df=mut,
    )
    prot_payload = load_or_build_prot_raw_fold_cache(
        seed=seed,
        arm=arm,
        fold_i=fold_i,
        train_cells=train_cells,
        eligible_cells=eligible_cells,
        prot_df=prot_core.loc[eligible_cells],
    )

    pairs = drug_to_pairs[str(drug)].copy()
    pairs = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].groupby("depmap_id", as_index=False)["y"].mean()
    y_by_cell = dict(zip(pairs["depmap_id"].astype(str), pairs["y"].astype(float)))
    keep_train = [c for c in train_cells if c in y_by_cell]
    keep_test = [c for c in test_cells if c in y_by_cell]
    if len(keep_train) < 20 or len(keep_test) < 5:
        raise RuntimeError("Selected interpretability case has too few train or test samples.")

    c2r = {c: i for i, c in enumerate(eligible_cells)}
    train_rows = np.array([c2r[c] for c in keep_train], dtype=int)
    test_rows = np.array([c2r[c] for c in keep_test], dtype=int)
    feat_keys = parse_feature_set(feature_set)
    train_arrays = build_modality_arrays({"rna": base_payload["Xr"], "cnv": base_payload["Xc"], "mut": base_payload["Xm"]}, prot_payload, train_rows, feat_keys)
    test_arrays = build_modality_arrays({"rna": base_payload["Xr"], "cnv": base_payload["Xc"], "mut": base_payload["Xm"]}, prot_payload, test_rows, feat_keys)
    y_train = np.array([y_by_cell[c] for c in keep_train], dtype=float)
    y_test = np.array([y_by_cell[c] for c in keep_test], dtype=float)

    add_indicators = "indicator" in strategy_name
    model = TaskAwareJointModel(
        d_rna=train_arrays["xr"].shape[1],
        d_cnv=train_arrays["xc"].shape[1],
        d_mut=train_arrays["xm"].shape[1],
        d_prot=train_arrays["xp"].shape[1],
        latent_dim=JOINT_LATENT_DIM,
        hidden_dim=JOINT_HIDDEN_DIM,
        dropout=JOINT_DROPOUT,
        add_indicators=add_indicators,
        use_modalities=feat_keys,
    )
    trainer = TaskAwareTrainer(
        model=model,
        pred_weight=JOINT_PRED_LOSS_WEIGHT,
        recon_weight=JOINT_RECON_LOSS_WEIGHT if "prot" in feat_keys else 0.0,
        remask_frac=JOINT_REMASK_FRAC if "prot" in feat_keys else 0.0,
        lr=JOINT_LR,
        weight_decay=JOINT_WEIGHT_DECAY,
        epochs=JOINT_EPOCHS,
        patience=JOINT_PATIENCE,
        batch_size=JOINT_BATCH_SIZE,
        device=TORCH_DEVICE,
    )
    fit_meta = trainer.fit(
        xr_train=train_arrays["xr"],
        xc_train=train_arrays["xc"],
        xm_train=train_arrays["xm"],
        xp_train=train_arrays["xp"],
        mp_train=train_arrays["mp"],
        y_train=y_train,
        seed=seed,
    )
    pred_out = trainer.predict(
        xr=test_arrays["xr"],
        xc=test_arrays["xc"],
        xm=test_arrays["xm"],
        xp=test_arrays["xp"],
        mp=test_arrays["mp"],
    )
    return {
        "trainer": trainer,
        "model": trainer.model,
        "fit_meta": fit_meta,
        "base_payload": base_payload,
        "prot_payload": prot_payload,
        "eligible_cells": eligible_cells,
        "train_cells": keep_train,
        "test_cells": keep_test,
        "train_arrays": train_arrays,
        "test_arrays": test_arrays,
        "y_train": y_train,
        "y_test": y_test,
        "pred_test": pred_out["pred"],
        "pred_out_test": pred_out,
        "feature_set": feature_set,
        "arm": arm,
        "drug": drug,
        "split_name": split_name,
        "c2r": c2r,
    }


def build_concat_input_and_names(case: dict, use_indicators: bool) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    tr = case["train_arrays"]
    te = case["test_arrays"]
    feat_names = []
    train_parts = []
    test_parts = []

    if tr["xr"].shape[1] > 0:
        train_parts.append(tr["xr"])
        test_parts.append(te["xr"])
        feat_names.extend([f"rna_pca_{i:03d}" for i in range(tr["xr"].shape[1])])
    if tr["xc"].shape[1] > 0:
        train_parts.append(tr["xc"])
        test_parts.append(te["xc"])
        feat_names.extend([f"cnv_pca_{i:03d}" for i in range(tr["xc"].shape[1])])
    if tr["xm"].shape[1] > 0:
        train_parts.append(tr["xm"])
        test_parts.append(te["xm"])
        feat_names.extend([f"mut_pca_{i:03d}" for i in range(tr["xm"].shape[1])])
    if tr["xp"].shape[1] > 0:
        prot_cols = list(case["prot_payload"]["prot_cols"])
        train_parts.append(tr["xp"])
        test_parts.append(te["xp"])
        feat_names.extend([f"prot::{c}" for c in prot_cols])
        if use_indicators:
            train_parts.append(tr["mp"])
            test_parts.append(te["mp"])
            feat_names.extend([f"prot_missing::{c}" for c in prot_cols])

    Xtr = np.concatenate(train_parts, axis=1) if train_parts else zeros_block(len(case["y_train"]))
    Xte = np.concatenate(test_parts, axis=1) if test_parts else zeros_block(len(case["y_test"]))
    return Xtr, Xte, feat_names


class JointCaseWrapper(nn.Module):
    def __init__(self, model: TaskAwareJointModel, dims: Dict[str, int], use_indicators: bool):
        super().__init__()
        self.model = model
        self.dims = dims
        self.use_indicators = bool(use_indicators)

    def forward(self, x_concat: torch.Tensor) -> torch.Tensor:
        start = 0
        xr = x_concat[:, start : start + self.dims["xr"]]
        start += self.dims["xr"]
        xc = x_concat[:, start : start + self.dims["xc"]]
        start += self.dims["xc"]
        xm = x_concat[:, start : start + self.dims["xm"]]
        start += self.dims["xm"]
        xp = x_concat[:, start : start + self.dims["xp"]]
        start += self.dims["xp"]
        if self.use_indicators and self.dims["xp"] > 0:
            mp = x_concat[:, start : start + self.dims["xp"]]
        else:
            mp = torch.ones_like(xp) if self.dims["xp"] > 0 else torch.zeros((x_concat.shape[0], 0), device=x_concat.device)
        out = self.model(xr=xr, xc=xc, xm=xm, xp=xp * mp, mp=mp)
        return out["yhat"]


def try_run_pathway_enrichment(feature_names: List[str], out_dir: Path) -> None:
    genes = []
    for g in feature_names:
        s = str(g)
        if "::" in s:
            s = s.split("::", 1)[1]
        if "__" in s:
            s = s.split("__")[-1]
        if s and not s.startswith("missing") and not s.endswith("pca") and not s.startswith("pca_"):
            genes.append(s)
    genes = list(dict.fromkeys(genes))[:500]
    if len(genes) == 0:
        print("[pathway] No gene-like features available for enrichment.")
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        gp = GProfiler(return_dataframe=True)
        enr = gp.profile(organism="hsapiens", query=genes, sources=["REAC", "GO:BP"])
        if enr is not None and enr.shape[0] > 0:
            enr.to_csv(out_dir / "pathway_enrichment_gprofiler.csv", index=False)
            print(f"[pathway] g:Profiler results written to {out_dir / 'pathway_enrichment_gprofiler.csv'}")
            return
    except Exception as e:
        print(f"[pathway] g:Profiler unavailable: {e}")

    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=["Reactome_2022", "GO_Biological_Process_2023"],
            organism="Human",
            outdir=str(out_dir / "gseapy"),
            cutoff=0.5,
        )
        if hasattr(enr, "results") and enr.results is not None:
            enr.results.to_csv(out_dir / "pathway_enrichment_gseapy.csv", index=False)
            print(f"[pathway] gseapy results written to {out_dir / 'pathway_enrichment_gseapy.csv'}")
            return
    except Exception as e:
        print(f"[pathway] gseapy unavailable: {e}")

    print("[pathway] No enrichment backend available. Install gprofiler-official or gseapy.")


class TinyGATRegressor(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim, heads=2, concat=True, dropout=0.1)
        self.conv2 = GATConv(hidden_dim * 2, hidden_dim, heads=1, concat=True, dropout=0.1)
        self.lin = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        from torch_geometric.nn import GATConv  # noqa: F401
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        return self.lin(x).squeeze(-1)


# Interpretability block
interp_dir = OUT_REPORTS / "interpretability"
interp_dir.mkdir(parents=True, exist_ok=True)

try:
    chosen_seed, chosen_arm, chosen_model, chosen_feature_set, chosen_strategy, auto_drug = choose_interpretability_target(
        detail_df=bakeoff_df,
        summary_df=bakeoff_summary,
        arm=INTERP_ARM,
        model=INTERP_MODEL,
        feature_set=INTERP_FEATURE_SET,
        seed=ALL_SEEDS[0],
    )
    if INTERP_DRUG is None:
        INTERP_DRUG = auto_drug

    print("\nINTERPRETABILITY TARGET")
    print("  seed          :", chosen_seed)
    print("  arm           :", chosen_arm)
    print("  model         :", chosen_model)
    print("  feature_set   :", chosen_feature_set)
    print("  strategy      :", chosen_strategy)
    print("  compound_id   :", INTERP_DRUG)
    print("  fold          :", INTERP_FOLD)

    case = train_case_model(
        seed=chosen_seed,
        arm=chosen_arm,
        feature_set=chosen_feature_set,
        strategy_name=chosen_strategy,
        fold_i=INTERP_FOLD,
        drug=INTERP_DRUG,
    )

    # Predicted vs observed
    plt.figure(figsize=(5.5, 5.0))
    plt.scatter(case["y_test"], case["pred_test"], s=28)
    lo = float(min(np.min(case["y_test"]), np.min(case["pred_test"])))
    hi = float(max(np.max(case["y_test"]), np.max(case["pred_test"])))
    plt.plot([lo, hi], [lo, hi], linestyle="--")
    plt.title(
        f"Task-aware case study\n{INTERP_DRUG} | {chosen_arm} | {chosen_feature_set}\n"
        f"Spearman={spearman_corr(case['y_test'], case['pred_test']):.3f}, R²={r2_score(case['y_test'], case['pred_test']):.3f}"
    )
    plt.xlabel("Observed AUC")
    plt.ylabel("Predicted AUC")
    plt.tight_layout()
    plt.savefig(interp_dir / "pred_vs_obs.png", dpi=160, bbox_inches="tight")
    plt.close()

    # Attention weights
    gate_test = case["pred_out_test"]["gates"]
    gate_rows = []
    for k, v in gate_test.items():
        gate_rows.append({
            "modality": k,
            "mean_weight": float(np.mean(v)),
            "median_weight": float(np.median(v)),
            "std_weight": float(np.std(v)),
        })
    gate_df = pd.DataFrame(gate_rows).sort_values("mean_weight", ascending=False)
    gate_df.to_csv(interp_dir / "attention_weights_summary.csv", index=False)
    plot_bar(gate_df, "modality", "mean_weight", "Mean modality attention weight", interp_dir / "attention_weights_summary.png", top_n=len(gate_df))

    # SHAP on a fold-safe surrogate over selected original features
    shap_feature_rank = None
    feat_keys = parse_feature_set(chosen_feature_set)
    all_case_cells = case["train_cells"] + case["test_cells"]
    X_train_parts = []
    X_all_parts = []
    feature_names = []

    if "rna" in feat_keys:
        Xr_tr, Xr_all, cols = build_original_block_filtered(
            rna,
            train_cells=case["train_cells"],
            all_cells=all_case_cells,
            top_k=INTERP_TOP_FEATURES_PER_MODALITY["rna"],
            strategy="median",
            do_scale=True,
        )
        X_train_parts.append(Xr_tr)
        X_all_parts.append(Xr_all)
        feature_names.extend([f"rna::{c}" for c in cols])

    if "cnv" in feat_keys:
        Xc_tr, Xc_all, cols = build_original_block_filtered(
            cnv,
            train_cells=case["train_cells"],
            all_cells=all_case_cells,
            top_k=INTERP_TOP_FEATURES_PER_MODALITY["cnv"],
            strategy="median",
            do_scale=True,
        )
        X_train_parts.append(Xc_tr)
        X_all_parts.append(Xc_all)
        feature_names.extend([f"cnv::{c}" for c in cols])

    if "mut" in feat_keys:
        Xm_tr, Xm_all, cols = build_original_block_filtered(
            mut,
            train_cells=case["train_cells"],
            all_cells=all_case_cells,
            top_k=INTERP_TOP_FEATURES_PER_MODALITY["mut"],
            strategy="most_frequent",
            do_scale=False,
        )
        X_train_parts.append(Xm_tr)
        X_all_parts.append(Xm_all)
        feature_names.extend([f"mut::{c}" for c in cols])

    if "prot" in feat_keys and case["train_arrays"]["xp"].shape[1] > 0:
        pred_all = case["trainer"].predict(
            xr=np.concatenate([case["train_arrays"]["xr"], case["test_arrays"]["xr"]], axis=0),
            xc=np.concatenate([case["train_arrays"]["xc"], case["test_arrays"]["xc"]], axis=0),
            xm=np.concatenate([case["train_arrays"]["xm"], case["test_arrays"]["xm"]], axis=0),
            xp=np.concatenate([case["train_arrays"]["xp"], case["test_arrays"]["xp"]], axis=0),
            mp=np.concatenate([case["train_arrays"]["mp"], case["test_arrays"]["mp"]], axis=0),
        )
        recon_all = pred_all["recon"]
        Xp_all_imp = np.concatenate([case["train_arrays"]["xp"], case["test_arrays"]["xp"]], axis=0).copy()
        Mp_all = np.concatenate([case["train_arrays"]["mp"], case["test_arrays"]["mp"]], axis=0) > 0.5
        if recon_all is not None:
            Xp_all_imp[~Mp_all] = recon_all[~Mp_all]
        Xp_tr_imp = Xp_all_imp[: len(case["train_cells"])]
        prot_cols = list(case["prot_payload"]["prot_cols"])
        var = np.var(Xp_tr_imp, axis=0)
        order = np.argsort(-np.nan_to_num(var, nan=-1.0))[: INTERP_TOP_FEATURES_PER_MODALITY["prot"]]
        X_train_parts.append(Xp_tr_imp[:, order])
        X_all_parts.append(Xp_all_imp[:, order])
        feature_names.extend([f"prot::{prot_cols[i]}" for i in order])

    if len(X_train_parts) > 0:
        Xtr_surr = np.concatenate(X_train_parts, axis=1)
        Xall_surr = np.concatenate(X_all_parts, axis=1)
        Xte_surr = Xall_surr[len(case["train_cells"]) :]
        ytr_surr = case["y_train"]

        surr = ExtraTreesRegressor(
            n_estimators=400,
            random_state=chosen_seed,
            n_jobs=-1,
            min_samples_leaf=2,
        )
        surr.fit(Xtr_surr, ytr_surr)
        try:
            explainer = shap.TreeExplainer(surr)
            shap_values = explainer.shap_values(Xte_surr)
            shap_abs = np.abs(np.asarray(shap_values)).mean(axis=0)
            shap_feature_rank = pd.DataFrame({
                "feature": feature_names,
                "mean_abs_shap": shap_abs,
            }).sort_values("mean_abs_shap", ascending=False)
            shap_feature_rank.to_csv(interp_dir / "shap_feature_rank.csv", index=False)
            plot_bar(
                shap_feature_rank.assign(label=shap_feature_rank["feature"]),
                "label",
                "mean_abs_shap",
                "Top SHAP features from case-study surrogate",
                interp_dir / "shap_feature_rank.png",
                top_n=20,
            )
            print(f"[shap] wrote {interp_dir / 'shap_feature_rank.csv'}")
        except Exception as e:
            print(f"[shap] unavailable: {e}")

    # Integrated Gradients on the joint model
    try:
        use_ind = "indicator" in chosen_strategy
        Xtr_concat, Xte_concat, ig_feature_names = build_concat_input_and_names(case, use_indicators=use_ind)
        dims = {
            "xr": case["train_arrays"]["xr"].shape[1],
            "xc": case["train_arrays"]["xc"].shape[1],
            "xm": case["train_arrays"]["xm"].shape[1],
            "xp": case["train_arrays"]["xp"].shape[1],
        }
        wrapper = JointCaseWrapper(case["model"], dims=dims, use_indicators=use_ind).to(TORCH_DEVICE)
        wrapper.eval()
        ig = IntegratedGradients(wrapper)
        x_test_t = torch.from_numpy(Xte_concat.astype(np.float32)).to(TORCH_DEVICE)
        baseline = torch.zeros((1, Xte_concat.shape[1]), dtype=torch.float32, device=TORCH_DEVICE)
        ig_attr = ig.attribute(x_test_t, baselines=baseline, n_steps=50)
        ig_rank = pd.DataFrame({
            "feature": ig_feature_names,
            "mean_abs_ig": np.abs(ig_attr.detach().cpu().numpy()).mean(axis=0),
        }).sort_values("mean_abs_ig", ascending=False)
        ig_rank.to_csv(interp_dir / "ig_feature_rank.csv", index=False)
        plot_bar(
            ig_rank.assign(label=ig_rank["feature"]),
            "label",
            "mean_abs_ig",
            "Top Integrated Gradients features",
            interp_dir / "ig_feature_rank.png",
            top_n=20,
        )
        print(f"[ig] wrote {interp_dir / 'ig_feature_rank.csv'}")
    except Exception as e:
        print(f"[ig] unavailable: {e}")

    # GNNExplainer on an optional graph surrogate over fused embeddings
    try:
        full_pred = case["trainer"].predict(
            xr=np.concatenate([case["train_arrays"]["xr"], case["test_arrays"]["xr"]], axis=0),
            xc=np.concatenate([case["train_arrays"]["xc"], case["test_arrays"]["xc"]], axis=0),
            xm=np.concatenate([case["train_arrays"]["xm"], case["test_arrays"]["xm"]], axis=0),
            xp=np.concatenate([case["train_arrays"]["xp"], case["test_arrays"]["xp"]], axis=0),
            mp=np.concatenate([case["train_arrays"]["mp"], case["test_arrays"]["mp"]], axis=0),
        )
        X_graph = full_pred["fused"]
        y_graph = np.concatenate([case["y_train"], case["y_test"]], axis=0).astype(np.float32)
        n_train_graph = len(case["y_train"])

        nbrs = NearestNeighbors(n_neighbors=min(GNN_KNN_K + 1, len(X_graph))).fit(X_graph)
        knn_idx = nbrs.kneighbors(X_graph, return_distance=False)
        edges = []
        for i in range(len(X_graph)):
            for j in knn_idx[i, 1:]:
                edges.append((i, int(j)))
                edges.append((int(j), i))
        edge_index = torch.tensor(np.array(edges).T, dtype=torch.long)
        data = Data(
            x=torch.tensor(X_graph, dtype=torch.float32),
            edge_index=edge_index,
            y=torch.tensor(y_graph, dtype=torch.float32),
        )
        train_mask = torch.zeros(len(X_graph), dtype=torch.bool)
        test_mask = torch.zeros(len(X_graph), dtype=torch.bool)
        train_mask[:n_train_graph] = True
        test_mask[n_train_graph:] = True
        data.train_mask = train_mask
        data.test_mask = test_mask

        gat = TinyGATRegressor(in_dim=X_graph.shape[1]).to(TORCH_DEVICE)
        data = data.to(TORCH_DEVICE)
        opt = torch.optim.Adam(gat.parameters(), lr=1e-3, weight_decay=1e-5)
        for _ in range(120):
            gat.train()
            pred = gat(data.x, data.edge_index)
            loss = F.mse_loss(pred[data.train_mask], data.y[data.train_mask])
            opt.zero_grad()
            loss.backward()
            opt.step()
        gat.eval()

        test_idx = torch.where(data.test_mask)[0]
        target_index = int(test_idx[0].item())
        explainer = Explainer(
            model=gat,
            algorithm=GNNExplainer(epochs=200),
            explanation_type="model",
            node_mask_type="attributes",
            edge_mask_type="object",
            model_config=dict(mode="regression", task_level="node", return_type="raw"),
        )
        explanation = explainer(data.x, data.edge_index, index=target_index)
        if hasattr(explanation, "node_mask") and explanation.node_mask is not None:
            np.save(interp_dir / "gnn_node_mask.npy", explanation.node_mask.detach().cpu().numpy())
        if hasattr(explanation, "edge_mask") and explanation.edge_mask is not None:
            np.save(interp_dir / "gnn_edge_mask.npy", explanation.edge_mask.detach().cpu().numpy())
        print(f"[gnnexplainer] wrote graph explanation artefacts to {interp_dir}")
    except Exception as e:
        print(f"[gnnexplainer] unavailable: {e}")

    # Pathway enrichment from SHAP first, then IG fallback
    top_feature_names: List[str] = []
    if shap_feature_rank is not None and not shap_feature_rank.empty:
        top_feature_names = shap_feature_rank["feature"].head(100).tolist()
    else:
        ig_path = interp_dir / "ig_feature_rank.csv"
        if ig_path.exists():
            ig_rank = pd.read_csv(ig_path)
            top_feature_names = ig_rank["feature"].head(100).tolist()

    try_run_pathway_enrichment(top_feature_names, interp_dir / "pathway_enrichment")

except Exception as e:
    print(f"[interpretability] skipped due to error: {e}")
